In [1]:
import sqlite3
import pandas as pd
import numpy as np
conn = sqlite3.connect('../laptops.db')

sql_query = """
    select L.Company, L.TypeName, S.ScreenResolution, S.Inches, S.Cpu, S.Ram, S.Memory, S.Gpu, S.Weight, S.OpSys, S.Price
    from Laptops L
    join Specs S on L.laptop_id = S.laptop_id
"""
df = pd.read_sql_query(sql_query, conn)


In [46]:
df_org = pd.read_sql_query(sql_query, conn)

In [2]:
# ram
df['Ram'] = df['Ram'].str.replace('GB', '')
df['Ram'] = df['Ram'].astype(int)

In [3]:
#processing (weight)
df['Weight'] = df['Weight'].str.replace('kg', '')
df['Weight'] = df['Weight'].astype(float)

In [4]:
# memory 
drive_type = ['SSD', 'HDD', 'Flash Storage', 'Hybrid']
def memory_extract(column, drive):
    extracted = column.str.extract(rf'(\d+)(GB|TB)\s+{drive}')
    size = pd.to_numeric(extracted[0])
    multiplier = extracted[1].map({'GB':1, 'TB':1024})
    df[f'{drive}_Size'] = (size * multiplier).fillna(0).astype(int)

for drive in drive_type:
    memory_extract(df['Memory'], drive)

In [5]:
# screen resolution 
df['TouchScreen'] = df['ScreenResolution'].str.contains('Touchscreen', case=False).astype(int)
df['IPS'] = df['ScreenResolution'].str.contains('IPS', case=False).astype(int)

In [6]:
xy_resolution = df['ScreenResolution'].str.extract(r'(\d+)x(\d+)')
x_res = xy_resolution[0].astype(int)
y_res = xy_resolution[1].astype(int)
df['PPI'] = np.sqrt((x_res)**2 + (y_res)**2) / df['Inches'].astype(float)

In [7]:
# company
count = df['Company'].value_counts()
other = count[count <= 10].index
df['Company'] = df['Company'].replace(other, 'Other')

In [8]:
df = pd.get_dummies(df, columns=['Company'], drop_first=True, dtype=int)

In [9]:
df = df.drop(columns=['ScreenResolution', 'Inches', 'Memory'])

In [10]:
def fetch_cpu(text):
    words = text.split()
    header = " ".join(words[:3])
    if header == 'Intel Core i7' or header == 'Intel Core i5' or header == 'Intel Core i3':
        return header
    else:
        if words[0] == 'Intel':
            return 'Other Intel Processor'
        else:
            return 'AMD Processor'
        
df['Cpu brand'] = df['Cpu'].apply(fetch_cpu)

In [11]:
df['Gpu_brand'] = df['Gpu'].apply(lambda x: x.split()[0])
df = df[df['Gpu_brand'] != 'ARM']

In [12]:
df = df.drop(columns=['Cpu','Gpu'])

In [13]:
def catOS(text):
    if "Windows" in text:
        return "Windows"
    elif "Mac" in text or "mac" in text:
        return "MacOS"
    else:
        return 'Others/No OS/Linux'

df['OpSys'] = df['OpSys'].apply(catOS)

In [14]:
df = pd.get_dummies(df, columns=['OpSys', 'Gpu_brand', 'Cpu brand'], drop_first=True, dtype=int)

In [16]:
df['TypeName'].unique()

<StringArray>
[         'Ultrabook',           'Notebook',            'Netbook',
             'Gaming', '2 in 1 Convertible',        'Workstation']
Length: 6, dtype: str